# Deep Dive: Implementing LLM Feedback Loops

## 📋 Summary

An **LLM Feedback Loop** is a system where the model's own output is evaluated —
either by code, by another model call, or by both — and the result is fed back
into a new generation attempt. The model iterates until it meets a quality threshold
or exhausts the retry budget.

### Why feedback loops matter
A single model call is a one-shot guess. Feedback loops let the model **correct itself**:

```
Generate → Test/Evaluate → [FAIL] → Generate again with feedback → Test again → [PASS] → Done
```

### Three feedback sources
| Source | Description | Best for |
|---|---|---|
| **Code execution** | Run the output and check for errors / test results | Code generation |
| **Rule-based evaluator** | Check against hard rules (length, format, keywords) | Structured outputs |
| **LLM critic** | Ask another model call to critique the output | Writing, reasoning |

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Single pass vs. iterative generation | 🟢 Beginner |
| 2 | Rule-based feedback | 🟢 Beginner |
| 3 | Code execution feedback | 🟡 Intermediate |
| 4 | LLM-as-critic feedback | 🟡 Intermediate |
| 5 | Self-critique (generator = critic) | 🔴 Advanced |
| 6 | Exercises | 🏋️ Practice |


## 0. Setup

In [2]:
# key input
import os
from google.colab import userdata
# set the key on colab on the left "key" icon (SECRETS)
# Automatically configures the key for SDKs to grab behind the scenes
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [8]:
# Install / import
from openai import OpenAI
from IPython.display import Markdown, display
import json, re

# ── API key ──────────────────────────────────────────────────────────────────

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"] ,
)

MODEL = "gpt-4.1-nano"

def chat(system: str, user: str, temperature: float = 0.3) -> str:
    """Single-turn helper. Returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content

def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))
    return Markdown(f"**{label}**\n\n{text}\n\n---").data

def chat_json(system: str, user: str) -> dict:
    """Like chat() but returns parsed JSON."""
    raw = chat(system, user, temperature=0.1)
    clean = raw.strip().strip('`').replace('json\n','',1).strip()
    return json.loads(clean)

In [7]:
from openai import OpenAI
from IPython.display import Markdown, display
import json, re, traceback, io
from contextlib import redirect_stdout, redirect_stderr


def extract_code(text: str) -> str:
    """Pull Python code from a markdown code block."""
    match = re.search(r'```python\s*(.*?)\s*```', text, re.DOTALL)
    return match.group(1) if match else text.strip()

def run_code(code_str: str) -> tuple[str, str, bool]:
    """
    Execute code_str safely.
    Returns (stdout, stderr, success).
    """
    stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
    namespace = {}
    try:
        with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exec(code_str, namespace)
        return stdout_buf.getvalue(), stderr_buf.getvalue(), True
    except Exception:
        return stdout_buf.getvalue(), traceback.format_exc(), False


## 1. Single Pass vs. Iterative Generation 🟢

The simplest demonstration: ask the model to write a function, run it, and if it fails,
ask it to fix the error. Just one retry to start.


In [9]:
TASK = """Write a Python function called `safe_divide(a, b)` that:
- Returns a / b
- Returns None if b is 0 (instead of raising ZeroDivisionError)
- Works with both integers and floats

After the function, add a few print statements to test it:
print(safe_divide(10, 2))   # Expected: 5.0
print(safe_divide(9, 3))    # Expected: 3.0
print(safe_divide(5, 0))    # Expected: None
print(safe_divide(7.5, 2.5)) # Expected: 3.0
"""

CODEGEN_SYSTEM = """You are a Python programmer. Write clean, correct code.
Return ONLY a Python code block (```python ... ```) with no explanation outside it."""

# Single pass
raw_response = chat(CODEGEN_SYSTEM, TASK)
code_v1 = extract_code(raw_response)

print("=== Generated Code ===")
print(code_v1)
print("\n=== Running Code ===")
stdout, stderr, success = run_code(code_v1)
print(f"Success: {success}")
if stdout: print("Output:", stdout)
if stderr: print("Error:", stderr)


=== Generated Code ===
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b

print(safe_divide(10, 2))   # Expected: 5.0
print(safe_divide(9, 3))    # Expected: 3.0
print(safe_divide(5, 0))    # Expected: None
print(safe_divide(7.5, 2.5)) # Expected: 3.0

=== Running Code ===
Success: True
Output: 5.0
3.0
None
3.0



In [10]:
# If it failed, one manual retry with the error injected
if not success:
    retry_system = CODEGEN_SYSTEM
    retry_user = f"""Original task:
{TASK}

Your previous attempt had this error:
{stderr}

Fix the code and return the corrected version."""

    raw_retry = chat(retry_system, retry_user)
    code_v2 = extract_code(raw_retry)

    print("=== Fixed Code ===")
    print(code_v2)
    print("\n=== Re-running ===")
    stdout2, stderr2, success2 = run_code(code_v2)
    print(f"Success: {success2}")
    if stdout2: print("Output:", stdout2)
    if stderr2: print("Error:", stderr2)
else:
    print("✅ First attempt succeeded — no retry needed!")


✅ First attempt succeeded — no retry needed!


## 2. Rule-Based Feedback 🟢

Rule-based feedback uses Python logic to evaluate outputs against measurable criteria.
No model call needed for evaluation — fast and deterministic.

**Task:** Generate a tweet thread about a topic, and enforce:
- Each tweet ≤ 280 characters
- Thread has exactly 5 tweets
- No hashtag spam (max 2 hashtags per tweet)
- First tweet must end with a hook (a question or "🧵" symbol)


In [11]:
def evaluate_tweet_thread(tweets: list[str]) -> tuple[bool, list[str]]:
    """Returns (all_pass, list_of_failures)."""
    failures = []

    if len(tweets) != 5:
        failures.append(f"Expected 5 tweets, got {len(tweets)}")

    for i, tweet in enumerate(tweets, 1):
        if len(tweet) > 280:
            failures.append(f"Tweet {i} is {len(tweet)} chars (max 280)")

        hashtag_count = tweet.count('#')
        if hashtag_count > 2:
            failures.append(f"Tweet {i} has {hashtag_count} hashtags (max 2)")

    if tweets and not (tweets[0].endswith('?') or '🧵' in tweets[0] or tweets[0].endswith('...')):
        failures.append("First tweet should end with '?', '...', or include '🧵'")

    return len(failures) == 0, failures


TWEET_SYSTEM = """You are a social media writer. Write exactly 5 tweets as a numbered list.
Format each tweet as:
1. [tweet text]
2. [tweet text]
...

Rules:
- Each tweet must be ≤ 280 characters
- Max 2 hashtags per tweet
- First tweet must hook the reader (end with '?' or include '🧵')"""

TOPIC = "How large language models actually work — the intuition behind transformers"

def parse_tweets(text: str) -> list[str]:
    """Parse numbered tweet list from response."""
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    tweets = []
    for line in lines:
        match = re.match(r'^\d+\.\s+(.*)', line)
        if match:
            tweets.append(match.group(1).strip())
    return tweets

def generate_tweet_thread_with_rules(topic: str, max_attempts: int = 4) -> list[str]:
    """Generate a tweet thread, retrying until rules pass or max_attempts reached."""
    user_prompt = f"Topic: {topic}"

    for attempt in range(1, max_attempts + 1):
        response = chat(TWEET_SYSTEM, user_prompt, temperature=0.7)
        tweets = parse_tweets(response)

        passed, failures = evaluate_tweet_thread(tweets)

        print(f"Attempt {attempt}: {'✅ PASS' if passed else f'❌ FAIL ({len(failures)} issues)'}")
        for f in failures:
            print(f"  - {f}")

        if passed:
            return tweets

        # Inject failures into next prompt
        failure_text = "\n".join(f"- {f}" for f in failures)
        user_prompt = f"""Topic: {topic}

Your previous attempt had these rule violations:
{failure_text}

Fix ALL violations and try again."""

    print("\n⚠️ Max attempts reached — returning last result anyway")
    return tweets

tweets = generate_tweet_thread_with_rules(TOPIC)
print("\n=== Final Thread ===")
for i, t in enumerate(tweets, 1):
    print(f"{i}. ({len(t)} chars) {t}")


Attempt 1: ✅ PASS

=== Final Thread ===
1. (170 chars) Ever wondered how giant language models understand and generate text? 🤔 Let’s break down the magic behind transformers and their incredible intuition! 🧵 #AI #Transformers
2. (210 chars) At their core, transformers process words as numbers, but not just any numbers—they focus on the relationships between words. Think of it as understanding context rather than just memorizing sentences. #NLP #AI
3. (197 chars) Self-attention is the secret sauce: it lets the model weigh the importance of each word relative to others in a sentence. So, “bank” in “river bank” and “financial bank” get different attention! 🌊💰
4. (215 chars) These models learn patterns by reading vast amounts of text, adjusting their internal ‘weights’ to predict the next word. It’s like a supercharged autocomplete, but with a deep understanding of language nuances. ✍️🤖
5. (214 chars) In essence, transformers mimic a kind of human-like intuition—spotting context, recognizi

## 3. Code Execution Feedback 🟡

The most powerful feedback loop for code generation: actually run the tests.

We'll build a complete test-driven code generation loop.


In [12]:
# Test suite for a data processing function
TEST_SUITE = [
    ("basic average",   "[1,2,3,4,5]",       "average", 3.0),
    ("basic sum",       "[1,2,3,4,5]",       "sum",     15),
    ("ignore strings",  "[1,'a',2,'b',3]",   "sum",     6),
    ("median odd",      "[3,1,2]",           "median",  2.0),
    ("median even",     "[1,2,3,4]",         "median",  2.5),
    ("empty list",      "[]",                "average", None),
    ("empty sum",       "[]",                "sum",     None),
    ("all non-numeric", "['a','b']",         "sum",     None),
]

def build_test_code(tests: list) -> str:
    lines = [
        "results = []",
        "passed = 0",
    ]
    for name, data, mode, expected in tests:
        lines.append(
            f"r = process_data({data}, mode='{mode}'); "
            f"ok = r == {expected!r}; "
            f"results.append(('{name}', ok, r, {expected!r})); "
            f"passed += ok"
        )
    lines.append("print(f'Passed: {passed}/{len(results)}')")
    lines.append("for name, ok, got, exp in results:")
    lines.append("    print(f'  {"✅" if ok else "❌"} {name}: got={got!r} expected={exp!r}')")
    return "\n".join(lines)

TEST_CODE_SUFFIX = build_test_code(TEST_SUITE)

CODEGEN_SYSTEM_V2 = """You are an expert Python developer.
Write a function called `process_data(data, mode='average')` that:
- Accepts a list (may contain non-numeric values) and a mode string
- mode='average': returns the mean of all numeric values, or None if no numbers
- mode='sum': returns the sum of all numeric values, or None if no numbers
- mode='median': returns the median of all numeric values, or None if no numbers
- Ignores non-numeric values silently

Return ONLY a ```python ... ``` code block with no explanation."""

def codegen_loop(task_desc: str, test_code: str, max_iter: int = 5) -> str:
    """Full code generation loop with test-based feedback."""
    user_prompt = task_desc
    best_code = ""

    for iteration in range(1, max_iter + 1):
        print(f"\n── Iteration {iteration} ──────────────────────────────────")

        # Generate
        raw = chat(CODEGEN_SYSTEM_V2, user_prompt, temperature=0.2)
        function_code = extract_code(raw)

        # Run with tests appended
        full_code = function_code + "\n\n" + test_code
        stdout, stderr, success = run_code(full_code)

        print(stdout.strip() if stdout else "(no output)")
        if stderr:
            print("Errors:", stderr[:200])

        # Did all tests pass?
        if "Passed:" in stdout:
            match = re.search(r'Passed: (\d+)/(\d+)', stdout)
            if match:
                passed, total = int(match.group(1)), int(match.group(2))
                if passed == total:
                    print(f"\n🎉 All {total} tests passed on iteration {iteration}!")
                    return function_code

        best_code = function_code

        # Build feedback for next iteration
        feedback_parts = []
        if not success:
            feedback_parts.append(f"Runtime error:\n{stderr}")
        if stdout:
            feedback_parts.append(f"Test results:\n{stdout}")

        user_prompt = f"""Previous attempt (iteration {iteration}) failed some tests.

Feedback:
{chr(10).join(feedback_parts)}

Fix the function so ALL tests pass. Return only the corrected ```python ... ``` block."""

    print("\n⚠️ Reached max iterations. Returning best attempt.")
    return best_code

final_code = codegen_loop(
    "Implement the process_data function as specified.",
    TEST_CODE_SUFFIX,
    max_iter=5
)

print("\n=== Final Function ===")
print(final_code)


SyntaxError: invalid character '✅' (U+2705) (3365640549.py, line 27)

## 4. LLM-as-Critic Feedback 🟡

When the output is prose (emails, essays, summaries), you can't run it.
Instead, use a **separate model call** as a critic.

The critic evaluates against a rubric and returns structured feedback,
which the generator uses on the next iteration.


In [ ]:
ESSAY_TASK = """Write a 200-word introductory paragraph explaining what
artificial intelligence is to a complete non-technical audience (imagine a retiree).
The paragraph must:
- Use zero jargon (no "neural networks", "algorithms", "models", "training")
- Use at least one vivid analogy
- End with a hook that makes the reader want to continue
"""

GENERATOR_SYSTEM = """You are a science writer who specializes in making AI topics
accessible to non-technical readers. Write clear, jargon-free prose."""

CRITIC_SYSTEM = """You are a strict writing editor. Evaluate the paragraph against these criteria:

1. jargon_free (true/false): Does it avoid ALL technical jargon? (neural networks, algorithms,
   models, training, parameters, weights, etc.)
2. has_analogy (true/false): Does it include at least one vivid analogy or metaphor?
3. word_count_ok (true/false): Is it 180–220 words?
4. has_hook (true/false): Does the final sentence create curiosity or a desire to read more?

Return ONLY valid JSON:
{
  "jargon_free": true/false,
  "has_analogy": true/false,
  "word_count_ok": true/false,
  "has_hook": true/false,
  "overall_pass": true/false,
  "specific_issues": ["list of specific things to fix"],
  "word_count": <actual count>
}"""

def llm_critic_loop(task: str, max_iter: int = 4) -> str:
    user_prompt = task

    for i in range(1, max_iter + 1):
        print(f"\n── Iteration {i} ──────────────────────────────────")

        # Generate
        draft = chat(GENERATOR_SYSTEM, user_prompt, temperature=0.6)
        word_count = len(draft.split())
        print(f"Draft length: {word_count} words")

        # Critique
        critique_raw = chat(CRITIC_SYSTEM, f"Paragraph to evaluate:\n{draft}", temperature=0.1)
        try:
            clean = critique_raw.strip().strip('`').replace('json\n','',1).strip()
            critique = json.loads(clean)
        except Exception:
            critique = {"overall_pass": False, "specific_issues": ["Parse error in critique"]}

        # Report
        passed_criteria = [k for k, v in critique.items() if isinstance(v, bool) and v and k != "overall_pass"]
        failed_criteria = [k for k, v in critique.items() if isinstance(v, bool) and not v and k != "overall_pass"]
        print(f"Passed: {passed_criteria}")
        print(f"Failed: {failed_criteria}")

        if critique.get("overall_pass"):
            print("\n✅ Passed all criteria!")
            return draft

        issues = "\n".join(f"- {issue}" for issue in critique.get("specific_issues", []))
        user_prompt = f"""Task: {task}

Your previous draft had these issues:
{issues}

Rewrite the paragraph fixing ALL issues. Return only the paragraph."""

    print("\n⚠️ Max iterations reached")
    return draft

final_paragraph = llm_critic_loop(ESSAY_TASK, max_iter=4)
show("Final Paragraph", final_paragraph)


## 5. Self-Critique Pattern 🔴

In the self-critique pattern, the **same model** is asked to:
1. Generate an output
2. Critique its own output
3. Revise based on the critique

This is cheaper (one model, not two) but can suffer from "blind spots" —
the model tends to critique the same things it generated, so it may miss systematic errors.

Compare it to the separate-critic approach by watching what it catches (or misses).


In [ ]:
SELF_CRITIQUE_SYSTEM = """You are a thoughtful writer and editor.

When asked to write something, use this process:
1. Write a first draft
2. Critically evaluate your draft against the stated requirements — be harsh
3. Rewrite based on your critique

Use this format:
<draft>
[your first draft]
</draft>

<critique>
[honest assessment: what works, what doesn't, specific issues]
</critique>

<revised>
[your improved version]
</revised>"""

response = chat(SELF_CRITIQUE_SYSTEM, ESSAY_TASK, temperature=0.6)
show("Self-Critique Process", response)

# Extract the final revised version
revised_match = re.search(r'<revised>(.*?)</revised>', response, re.DOTALL)
if revised_match:
    final = revised_match.group(1).strip()
    print(f"\nFinal word count: {len(final.split())}")


### 🧐 Comparison: Self-Critique vs. Separate Critic

| Dimension | Separate Critic | Self-Critique |
|---|---|---|
| Cost | 2× model calls per iteration | 1× model call per iteration |
| Independence | High — different context window | Low — same biases |
| Consistency | Reliable rubric enforcement | Variable — depends on prompt |
| Speed | Slower | Faster |
| Best for | Production pipelines | Quick drafts, exploration |

Which approach produced a better result for the essay task?


## 6. Exercises 🏋️


### Exercise 1 — Multi-Criteria Code Loop
Extend the `codegen_loop` to track:
- Which tests pass/fail per iteration
- Whether the number of passing tests is increasing
- If 2 consecutive iterations have the same failure pattern → stop (stuck in a loop)


In [ ]:
# TODO: Enhanced codegen loop with stuck-detection

def codegen_loop_v2(task_desc: str, test_code: str, max_iter: int = 5) -> str:
    """Like codegen_loop but detects if we're stuck."""
    previous_failures = set()

    for iteration in range(1, max_iter + 1):
        # TODO: generate, run, collect failures
        # TODO: if failures == previous_failures → print warning and break
        # TODO: update previous_failures
        pass

    return "TODO: implement"


### Exercise 2 — Mixed Feedback (Code + LLM)
Build a loop that:
1. Generates a Python function
2. Runs unit tests (code-based feedback)
3. If tests pass, ALSO runs an LLM quality check on the code style
   (is it readable? well-commented? handles edge cases elegantly?)
4. Only accepts the output if BOTH gates pass


In [ ]:
# TODO: mixed feedback loop
CODE_STYLE_CRITIC = """You are a code reviewer. Evaluate this Python function:
{
  "is_readable": true/false,
  "is_well_commented": true/false,
  "handles_edge_cases": true/false,
  "overall_pass": true/false,
  "improvements": ["..."]
}"""

def mixed_feedback_loop(task: str, tests: str, max_iter: int = 5) -> str:
    """Combines code execution feedback with LLM style feedback."""
    pass  # TODO


### Exercise 3 — Measure Loop Efficiency
Run the same code generation task 5 times with the full loop.
For each run, record:
- How many iterations it took to pass
- The final pass/fail state

Then compute: average iterations to success, success rate.

This simulates how you'd benchmark a feedback loop system.


In [ ]:
# TODO: Benchmark the codegen loop
import statistics

results = []
for trial in range(5):
    print(f"\n=== Trial {trial+1} ===")
    # TODO: run codegen_loop and track iterations
    # results.append({"trial": trial+1, "iterations": ..., "success": ...})

# TODO: print statistics
# print(f"Average iterations: {statistics.mean([r['iterations'] for r in results]):.1f}")
# print(f"Success rate: {sum(r['success'] for r in results)/len(results)*100:.0f}%")


## 🏁 Summary & Key Takeaways

| Feedback type | Evaluation | Best for | Reliability |
|---|---|---|---|
| Rule-based | Python code | Format, length, counts | 🟢 Deterministic |
| Code execution | Run + tests | Code generation | 🟢 Deterministic |
| LLM critic | Model call | Prose, reasoning | 🟡 Probabilistic |
| Self-critique | Same call | Quick drafts | 🟡 Probabilistic |
| Mixed | Code + LLM | Production quality | 🟢🟡 Both |

**Design principles:**
- Use code-based feedback whenever possible (fast, deterministic, free)
- Add LLM critics for subjective quality (style, tone, clarity)
- Always set a max iteration limit
- Detect "stuck" loops (same feedback twice → stop or change strategy)
- Log every iteration — you'll learn what the model consistently gets wrong

### 🎓 You've completed all 5 deep-dive notebooks!
Each concept builds on the previous:
Role-Based Prompting → Chain-of-Thought → Prompt Refinement → Chaining → Feedback Loops

Together, these are the core primitives of modern LLM-powered systems.
